In [1]:
#FDA Load + NDC11 + Status

import requests
import pandas as pd
import time

url = "https://api.fda.gov/drug/ndc.json"

# Get total count
meta = requests.get(url).json()
total = meta["meta"]["results"]["total"]
print(f"Total records: {total}")

# Pull everything in pages of 1000
records = []
skip = 0
limit = 1000

while skip < total:
    params = {"limit": limit, "skip": skip}
    response = requests.get(url, params=params)
    data = response.json()
    
    for item in data.get("results", []):
        ingredients = item.get("active_ingredients") or []
        first_ingredient = ingredients[0] if ingredients else {}
        
        packaging = item.get("packaging") or []
        first_package = packaging[0] if packaging else {}
        
        flat = {
            "product_ndc": item.get("product_ndc"),
            "generic_name": item.get("generic_name"),
            "brand_name": item.get("brand_name"),
            "labeler_name": item.get("labeler_name"),
            "dosage_form": item.get("dosage_form"),
            "route": ", ".join(item.get("route") or []),
            "marketing_category": item.get("marketing_category"),
            "listing_expiration_date": item.get("listing_expiration_date"),
            "product_type": item.get("product_type"),
            "finished": item.get("finished"),
            "active_ingredient": first_ingredient.get("name"),
            "strength": first_ingredient.get("strength"),
            "package_ndc": first_package.get("package_ndc"),
            "rxcui": ", ".join(item.get("openfda", {}).get("rxcui") or []),
        }
        records.append(flat)
    
    skip += limit
    time.sleep(0.2)
    print(f"Pulled {min(skip, total)} / {total}")

df = pd.DataFrame(records)
print(f"\nFinal row count: {len(df)}")

def to_ndc11(ndc, hyphenated=False):
    if not ndc or not isinstance(ndc, str):
        return None
    ndc = ndc.strip()
    parts = ndc.split("-")
    if len(parts) == 1 and len(ndc) == 11 and ndc.isdigit():
        parts = [ndc[:5], ndc[5:9], ndc[9:]]
    elif len(parts) != 3 or not all(p.isdigit() for p in parts):
        return None
    lengths = tuple(len(p) for p in parts)
    if lengths not in {(4, 4, 2), (5, 3, 2), (5, 4, 1), (5, 4, 2)}:
        return None
    labeler = parts[0].zfill(5)
    product = parts[1].zfill(4)
    package = parts[2].zfill(2)
    if hyphenated:
        return f"{labeler}-{product}-{package}"
    return labeler + product + package

df["ndc11"] = df["package_ndc"].apply(lambda x: to_ndc11(x) if pd.notna(x) else None)
df["package_ndc_status"] = df["package_ndc"].apply(
    lambda x: "Available" if pd.notna(x) else "Package not available"
)

print(df[["package_ndc", "ndc11", "package_ndc_status"]].head(10))
print(f"Rows with valid ndc11: {df['ndc11'].notna().sum():,}")

Total records: 138229
Pulled 1000 / 138229
Pulled 2000 / 138229
Pulled 3000 / 138229
Pulled 4000 / 138229
Pulled 5000 / 138229
Pulled 6000 / 138229
Pulled 7000 / 138229
Pulled 8000 / 138229
Pulled 9000 / 138229
Pulled 10000 / 138229
Pulled 11000 / 138229
Pulled 12000 / 138229
Pulled 13000 / 138229
Pulled 14000 / 138229
Pulled 15000 / 138229
Pulled 16000 / 138229
Pulled 17000 / 138229
Pulled 18000 / 138229
Pulled 19000 / 138229
Pulled 20000 / 138229
Pulled 21000 / 138229
Pulled 22000 / 138229
Pulled 23000 / 138229
Pulled 24000 / 138229
Pulled 25000 / 138229
Pulled 26000 / 138229
Pulled 27000 / 138229
Pulled 28000 / 138229
Pulled 29000 / 138229
Pulled 30000 / 138229
Pulled 31000 / 138229
Pulled 32000 / 138229
Pulled 33000 / 138229
Pulled 34000 / 138229
Pulled 35000 / 138229
Pulled 36000 / 138229
Pulled 37000 / 138229
Pulled 38000 / 138229
Pulled 39000 / 138229
Pulled 40000 / 138229
Pulled 41000 / 138229
Pulled 42000 / 138229
Pulled 43000 / 138229
Pulled 44000 / 138229
Pulled 45000 / 1382

In [2]:
#Run to expand for dataset
pd.set_option("display.max_colwidth", 40)
df.head(1)

,product_ndc,generic_name,brand_name,labeler_name,dosage_form,route,marketing_category,listing_expiration_date,product_type,finished,active_ingredient,strength,package_ndc,rxcui,ndc11,package_ndc_status
0,76162-119,Polyethylene Glycol 3350,Topcare ClearLax,Topco Associates LLC,"POWDER, FOR SOLUTION",ORAL,ANDA,20271231,HUMAN OTC DRUG,True,POLYETHYLENE GLYCOL 3350,17 g/17g,76162-119-02,"876193, 880407",76162011902,Available


In [3]:
#RxNorm NDC Crosswalk (from zip)

import pandas as pd
import zipfile
import requests
from pathlib import Path

url = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_11022026.zip"
zip_path = Path("RxNorm_full_prescribe_11022026.zip")

if not zip_path.exists():
    print("Downloading...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Unique NDCs:", ndc_xwalk["ndc11"].nunique())
print("Unique RxCUIs:", ndc_xwalk["rxcui"].nunique())

(239693, 2)
         ndc11  rxcui
0  59050026800  91348
1  00295117904  91349
2  00295117916  91349
3  00363026816  91349
4  00363026832  91349
Unique NDCs: 239693
Unique RxCUIs: 16713


In [5]:
#RxNav Concepts

import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

rxcuis = ndc_xwalk["rxcui"].dropna().unique().tolist()
print(f"Unique RxCUIs: {len(rxcuis):,}")

def fetch_concept(rxcui):
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            prop = data.get("properties", {})
            return {
                "rxcui": rxcui,
                "name": prop.get("name"),
                "tty": prop.get("tty"),
                "suppress": prop.get("suppress"),
            }
    except Exception:
        pass
    return {"rxcui": rxcui, "name": None, "tty": None, "suppress": None}

print("Fetching concepts from RxNav (parallel)...")
results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(fetch_concept, rxcui): rxcui for rxcui in rxcuis}
    for i, future in enumerate(as_completed(futures), 1):
        results.append(future.result())
        if i % 500 == 0:
            print(f"  {i:,} / {len(rxcuis):,}")

concepts = pd.DataFrame(results)
print(f"Done. Concepts retrieved: {len(concepts):,}")
print(concepts.head())

Unique RxCUIs: 16,713
Fetching concepts from RxNav (parallel)...
  500 / 16,713
  1,000 / 16,713
  1,500 / 16,713
  2,000 / 16,713
  2,500 / 16,713
  3,000 / 16,713
  3,500 / 16,713
  4,000 / 16,713
  4,500 / 16,713
  5,000 / 16,713
  5,500 / 16,713
  6,000 / 16,713
  6,500 / 16,713
  7,000 / 16,713
  7,500 / 16,713
  8,000 / 16,713
  8,500 / 16,713
  9,000 / 16,713
  9,500 / 16,713
  10,000 / 16,713
  10,500 / 16,713
  11,000 / 16,713
  11,500 / 16,713
  12,000 / 16,713
  12,500 / 16,713
  13,000 / 16,713
  13,500 / 16,713
  14,000 / 16,713
  14,500 / 16,713
  15,000 / 16,713
  15,500 / 16,713
  16,000 / 16,713
  16,500 / 16,713
Done. Concepts retrieved: 16,713
   rxcui                                     name  tty suppress
0  94346  desoximetasone 0.0005 MG/MG Topical ...  SBD        N
1  93253  hydrochlorothiazide 25 MG / triamter...  SBD        N
2  94345  desoximetasone 2.5 MG/ML Topical Cre...  SBD        N
3  92752  fluorouracil 10 MG/ML Topical Cream ...  SBD        N
4  92758 

In [7]:
# Join All Three (corrected)
# FDA → Crosswalk on ndc11, then → Concepts on the crosswalk's rxcui

joined = (
    df.merge(ndc_xwalk, on="ndc11", how="left", suffixes=("", "_xwalk"))
      .merge(concepts, left_on="rxcui_xwalk", right_on="rxcui", how="left", suffixes=("", "_concept"))
)

print(f"Joined shape: {joined.shape}")
print(f"Matched to RxNorm crosswalk: {joined['rxcui_xwalk'].notna().sum():,}")
print(f"Matched to concept name: {joined['name'].notna().sum():,}")
joined.head(10)

Joined shape: (26000, 21)
Matched to RxNorm crosswalk: 15,423
Matched to concept name: 15,390


,product_ndc,generic_name,brand_name,labeler_name,dosage_form,route,marketing_category,listing_expiration_date,product_type,finished,...,strength,package_ndc,rxcui,ndc11,package_ndc_status,rxcui_xwalk,rxcui_concept,name,tty,suppress
0,76162-119,Polyethylene Glycol 3350,Topcare ClearLax,Topco Associates LLC,"POWDER, FOR SOLUTION",ORAL,ANDA,20271231,HUMAN OTC DRUG,True,...,17 g/17g,76162-119-02,"876193, 880407",76162011902,Available,880407,880407,polyethylene glycol 3350 17000 MG Po...,SBD,N
1,76282-247,Levetiracetam,Levetiracetam,"Exelan Pharmaceuticals, Inc.",TABLET,ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,...,500 mg/1,76282-247-12,"311288, 311289, 311290",76282024712,Available,311289,311289,levetiracetam 500 MG Oral Tablet,SCD,N
2,76348-495,"Menthol, Methyl Salicylate",EXTREME EMU,"Renu Laboratories, LLC",CREAM,TOPICAL,OTC MONOGRAPH DRUG,20261231,HUMAN OTC DRUG,True,...,11.2 g/112g,76348-495-04,311503,76348049504,Available,311503,311503,menthol 20 MG/ML / methyl salicylate...,SCD,N
3,76385-112,Carvedilol,Carvedilol,"UNICHEM PHARMACEUTICALS (USA), INC.","TABLET, FILM COATED",ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,...,12.5 mg/1,76385-112-01,"200031, 200032, 200033, 686924",76385011201,Available,200032,200032,carvedilol 12.5 MG Oral Tablet,SCD,N
4,0002-7511,Insulin lispro,Humalog Mix75/25,Eli Lilly and Company,"INJECTION, SUSPENSION",SUBCUTANEOUS,BLA,20271231,HUMAN PRESCRIPTION DRUG,True,...,100 [iU]/mL,0002-7511-01,"259111, 752388, 847252, 847254",00002751101,Available,752388,752388,insulin lispro 25 UNT/ML / insulin l...,SBD,N
5,76420-674,PANTOPRAZOLE,PANTOPRAZOLE SODIUM,"Asclemed USA, Inc.","TABLET, DELAYED RELEASE",ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,...,40 mg/1,76420-674-01,"251872, 314200",76420067401,Available,314200,314200,pantoprazole 40 MG Delayed Release O...,SCD,N
6,0597-0405,adalimumab-adbm,Cyltezo,Boehringer Ingelheim Pharmaceuticals...,KIT,,BLA,20271231,HUMAN PRESCRIPTION DRUG,True,...,NaN,0597-0405-80,"797544, 2640883, 2640889, 2640891, 2...",00597040580,Available,2640894,2640894,0.4 ML adalimumab-adbm 50 MG/ML Pref...,SBD,N
7,76891-155,Alcohol,Hand Sanitizer Matcha Mood,Scent Theory Products LLC,SPRAY,TOPICAL,OTC MONOGRAPH DRUG,20261231,HUMAN OTC DRUG,True,...,62 mL/100mL,76891-155-30,1020364,76891015530,Available,NaN,NaN,NaN,NaN,NaN
8,77931-0001,"Fucus Vesiculosus, Chelidonium Majus...",Drop20,Pick Enterprises Corp,LIQUID,ORAL,UNAPPROVED HOMEOPATHIC,20271231,HUMAN OTC DRUG,True,...,4 [hp_X]/mL,77931-0001-1,,77931000101,Available,NaN,NaN,NaN,NaN,NaN
9,79794-189,antibacterial hand wipes,antibacterial hand wipes (Cherry),Zhejiang Furuisen Spunlaced Nonwoven...,CLOTH,CUTANEOUS,OTC MONOGRAPH DRUG,20271231,HUMAN OTC DRUG,True,...,.13 g/1,79794-189-01,1038558,79794018901,Available,1038558,1038558,benzalkonium chloride 1.3 MG/ML Medi...,SCD,N


In [9]:
# TTY lookup
tty_lookup = pd.DataFrame({
    "tty": [
        "SCD", "SBD", "GPCK", "BPCK",
        "IN", "PIN", "BN", "DF",
        "SCDF", "SBDF", "SCDG", "SBDG",
        "MIN", "TMSY", "SY", "PSN"
    ],
    "term_type": [
        "Semantic Clinical Drug",
        "Semantic Branded Drug",
        "Generic Pack",
        "Branded Pack",
        "Ingredient",
        "Precise Ingredient",
        "Brand Name",
        "Dose Form",
        "Semantic Clinical Drug Form",
        "Semantic Branded Drug Form",
        "Semantic Clinical Drug Group",
        "Semantic Branded Drug Group",
        "Multiple Ingredients",
        "Tall Man Synonym",
        "Synonym",
        "Prescribable Name"
    ]
})

# Attach meaning to the joined data
joined = joined.merge(tty_lookup, on="tty", how="left")

print(f"TTY values mapped: {joined['term_type'].notna().sum():,}")
print("\nSample TTY breakdown:")
print(joined["term_type"].value_counts(dropna=False).head(15))

TTY values mapped: 15,390

Sample TTY breakdown:
term_type
Semantic Clinical Drug    13619
NaN                       10610
Semantic Branded Drug      1494
Branded Pack                156
Generic Pack                121
Name: count, dtype: int64


In [ ]:
#File Export

from pathlib import Path

out_folder = Path(r"C:\Add_Path_Here")
out_folder.mkdir(parents=True, exist_ok=True)

# Final joined file
joined.to_parquet(out_folder / "fda_rxnorm_joined.parquet", index=False)
joined.to_csv(out_folder / "fda_rxnorm_joined.csv", index=False)

# Optional intermediate saves
ndc_xwalk.to_parquet(out_folder / "rxnorm_ndc_xwalk.parquet", index=False)
concepts.to_parquet(out_folder / "rxnav_concepts.parquet", index=False)

print("Exported.")

Exported.
